In [1]:
import pandas as pd
import random
from datetime import datetime, timedelta
import string


# Danh mục sản phẩm (Master Data)
catalog = {
    'DIOR-SAUVAGE-100ML': 3500000,
    'CHANEL-COCO-50ML': 2800000,
    'YSL-LIBRE-90ML': 3200000,
    'TF-OUDWOOD-50ML': 5500000,
    'LELABO-S33-100ML': 6000000,
    'MAC-RUBYWOO': 550000,
    'ESTEE-ANR-50ML': 2200000,
    'KIEHLS-UFC-125ML': 1500000
}
skus = list(catalog.keys())
start_date = datetime(2026, 1, 1)
end_date = datetime(2026, 10, 31)

def rand_date():
    return start_date + timedelta(seconds=random.randint(0, int((end_date - start_date).total_seconds())))

# 1. Nguồn Website (Magento 2)
web_data = []
for i in range(1, 5001):
    dt = rand_date()
    sku = random.choice(skus)
    qty = random.choices([1, 2, 3], weights=[0.8, 0.15, 0.05])[0]
    web_data.append({
        'order_id': f'WEB-{100000 + i}',
        'created_at': dt.strftime('%Y-%m-%dT%H:%M:%SZ'), # Chuẩn ISO 8601 UTC
        'status': random.choices(['pending', 'processing', 'complete', 'canceled', 'closed'], weights=[0.05, 0.1, 0.7, 0.1, 0.05])[0],
        'customer_id': f'CUST-{random.randint(1000, 9999)}',
        'sku': sku,
        'qty': qty,
        'price': catalog[sku],
        'utm_source': random.choices(['google', 'facebook', 'direct', 'email'], weights=[0.4, 0.3, 0.2, 0.1])[0],
        'utm_medium': random.choice(['cpc', 'organic', 'social', 'newsletter'])
    })
pd.DataFrame(web_data).to_csv('website_orders.csv', index=False)

# 2. Nguồn Shopee E-commerce
shopee_data = []
for i in range(1, 10001):
    dt = rand_date()
    sku = random.choice(skus)
    qty = random.choices([1, 2], weights=[0.9, 0.1])[0]
    # Cấu trúc mã đơn Shopee (VD: 261031AXYZ1234)
    order_sn = dt.strftime('%y%m%d') + ''.join(random.choices(string.ascii_uppercase, k=4)) + ''.join(random.choices(string.digits, k=4))
    original_price = catalog[sku]
    shopee_data.append({
        'order_sn': order_sn,
        'order_date': dt.strftime('%Y-%m-%d %H:%M:%S'), # Định dạng ngày khác biệt
        'order_status': random.choices(['UNPAID', 'READY_TO_SHIP', 'SHIPPED', 'COMPLETED', 'CANCELLED'], weights=[0.05, 0.05, 0.1, 0.7, 0.1])[0],
        'sku': f'SP-{sku}', # Nhân viên thêm tiền tố SP-
        'quantity': qty,
        'original_price': original_price,
        'shopee_fee': int(original_price * qty * 0.03), # 3% phí Shopee
        'payment_fee': int(original_price * qty * 0.02), # 2% phí thanh toán
        'shipping_subsidy': random.choice([0, 15000, 25000])
    })
pd.DataFrame(shopee_data).to_csv('shopee_orders.csv', index=False)

# 3. Nguồn POS (Cửa hàng Offline)
pos_data = []
stores = ['QN-01', 'HN-01', 'HCM-01']
for i in range(1, 8001):
    dt = rand_date()
    sku = random.choice(skus)
    qty = random.choices([1, 2, 3], weights=[0.7, 0.2, 0.1])[0]
    store = random.choice(stores)
    # 40% khách hàng không để lại số điện thoại tích điểm
    phone = f"09{random.randint(10000000, 99999999)}" if random.random() > 0.4 else ""
    pos_data.append({
        'transaction_id': f"{store}-{dt.strftime('%Y%m%d')}-{i:04d}",
        'timestamp': dt.strftime('%d/%m/%Y %H:%M:%S'), # Định dạng ngày kiểu Việt Nam
        'status': random.choices(['Thành công', 'Hoàn trả'], weights=[0.9, 0.1])[0],
        'store_id': store,
        'customer_phone': phone, # Có dòng Null/Empty
        'product_code': sku,
        'amount': qty,
        'total_value': catalog[sku] * qty
    })
pd.DataFrame(pos_data).to_csv('pos_transactions.csv', index=False)